# DeltaNet Diagnostics Notebook

这个 notebook 针对 Qwen3.5-4B 的 DeltaNet 层做 5 件事：

1. 抓 decode 阶段 DeltaNet 层的实际输入 / 输出 shape
2. 定位 DeltaNet forward 的 transformers 源码并保存一份副本
3. 检查 `past_key_values` 里 DeltaNet / Full Attention 混合 cache 的实际 state shape
4. 在安装 `fla` 前后，重新跑 phase1 端到端 baseline 做性能对比
5. 定位 `fla.ops.gated_delta_rule.fused_recurrent_gated_delta_rule` 的源码并保存副本

所有可复用产物都会保存到 `artifacts/deltanet_diagnostics/`。

In [1]:
import importlib.util
import inspect
import json
import platform
import subprocess
import sys
import textwrap
from pathlib import Path
from pprint import pprint

import accelerate
import torch
import transformers

from deltanet_diagnostics import (
    capture_forward_io,
    find_first_deltanet_layer,
    list_module_tensors,
    summarize_past_key_values,
)
from phase1_utils import MODEL_DIR, build_baseline_rows, load_model_and_tokenizer

ROOT_DIR = Path.cwd()
ARTIFACT_DIR = ROOT_DIR / 'artifacts' / 'deltanet_diagnostics'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('python:', platform.python_version())
print('python_executable:', sys.executable)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))
print('model_dir:', MODEL_DIR)
print('artifact_dir:', ARTIFACT_DIR.resolve())
print('initial_fla_available:', importlib.util.find_spec('fla') is not None)


/home/haozhong/vllm-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python: 3.12.3
python_executable: /home/haozhong/vllm-env/bin/python3
torch: 2.9.1+cu128
transformers: 5.6.0.dev0
accelerate: 1.13.0
cuda_available: True
cuda_device: NVIDIA GeForce RTX 4090 Laptop GPU
model_dir: /home/haozhong/ECE9483/models/Qwen3.5-4B
artifact_dir: /home/haozhong/ECE9483/artifacts/deltanet_diagnostics
initial_fla_available: True


## Load Model

In [2]:
model, tokenizer, config = load_model_and_tokenizer(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map='auto',
)
model_device = next(model.parameters()).device

print('config_class:', type(config).__name__)
print('model_class:', type(model).__name__)
print('model_device:', model_device)
print('architectures:', getattr(config, 'architectures', None))
text_config = getattr(config, 'text_config', None)
if text_config is not None:
    print('text_model_type:', getattr(text_config, 'model_type', None))
    print('num_hidden_layers:', getattr(text_config, 'num_hidden_layers', None))
    print('layer_types_head:', getattr(text_config, 'layer_types', None)[:8])


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/723 [00:00<02:40,  4.51it/s]

Loading weights:   6%|▌         | 42/723 [00:00<00:04, 163.12it/s]

Loading weights:  13%|█▎        | 93/723 [00:00<00:02, 287.23it/s]

...

Loading weights:  55%|█████▍    | 396/723 [00:01<00:00, 364.89it/s]

Loading weights:  69%|██████▊   | 496/723 [00:01<00:00, 543.29it/s]

Loading weights:  99%|█████████▉| 716/723 [00:01<00:00, 997.52it/s]

Loading weights: 100%|██████████| 723/723 [00:01<00:00, 513.23it/s]

config_class: Qwen3_5Config
model_class: Qwen3_5ForConditionalGeneration
model_device: cuda:0
architectures: ['Qwen3_5ForConditionalGeneration']
text_model_type: qwen3_5_text
num_hidden_layers: 32
layer_types_head: ['linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention']


## 1. DeltaNet Layer Structure

In [3]:
deltanet_info = find_first_deltanet_layer(model)
deltanet_layer = deltanet_info['module']
module_tensor_rows = list_module_tensors(deltanet_layer)

structure_path = ARTIFACT_DIR / 'deltanet_layer_structure.txt'
tensor_summary_path = ARTIFACT_DIR / 'deltanet_layer_tensors.json'
structure_path.write_text(str(deltanet_layer), encoding='utf-8')
tensor_summary_path.write_text(json.dumps(module_tensor_rows, indent=2), encoding='utf-8')

print('resolved_attr_path:', deltanet_info['attr_path'])
print('resolved_module_name:', deltanet_info['module_name'])
print()
print(deltanet_layer)
print()
for row in module_tensor_rows:
    prefix = '[buffer] ' if row['kind'] == 'buffer' else ''
    print(f"  {prefix}{row['name']}: {row['shape']} {row['dtype']}")
print()
print('saved_structure:', structure_path.resolve())
print('saved_tensor_summary:', tensor_summary_path.resolve())


resolved_attr_path: model.language_model.layers[0].linear_attn
resolved_module_name: model.language_model.layers.0.linear_attn

Qwen3_5GatedDeltaNet(
  (act): SiLUActivation()
  (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
  (norm): FusedRMSNormGated(128, eps=1e-06, activation=silu)
  (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
  (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
  (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
  (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
  (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
)

  dt_bias: [32] torch.float16
  A_log: [32] torch.float16
  conv1d.weight: [8192, 1, 4] torch.float16
  norm.weight: [128] torch.float16
  out_proj.weight: [2560, 4096] torch.float16
  in_proj_qkv.weight: [8192, 2560] torch.float16
  in_proj_z.weight: [4096, 2560] torch.float16
  in_proj_b.weight: [32, 256

## 2. DeltaNet Forward Source

In [4]:
deltanet_source_path = Path(inspect.getfile(type(deltanet_layer)))
deltanet_source_text = deltanet_source_path.read_text(encoding='utf-8')
deltanet_source_copy = ARTIFACT_DIR / 'transformers_qwen3_5_modeling_qwen3_5.py'
deltanet_source_copy.write_text(deltanet_source_text, encoding='utf-8')

print('deltanet_source_path:', deltanet_source_path)
print('saved_copy:', deltanet_source_copy.resolve())
print('source_line_count:', len(deltanet_source_text.splitlines()))
print()
for line in deltanet_source_text.splitlines()[:160]:
    print(line)


deltanet_source_path: /home/haozhong/vllm-env/lib/python3.12/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py
saved_copy: /home/haozhong/ECE9483/artifacts/deltanet_diagnostics/transformers_qwen3_5_modeling_qwen3_5.py
source_line_count: 2181

#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
#           This file was automatically generated from src/transformers/models/qwen3_5/modular_qwen3_5.py.
#               Do NOT edit this file manually as any edits will be overwritten by the generation of
#             the file from the modular. If any change should be done, please apply the change to the
#                          modular_qwen3_5.py file directly. One of our CI enforces this.
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
# Copyright 2025 The Qwen Team and The HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the Lice

## 3. Decode Hook And Cache Shape

In [5]:
prompt = 'Hello'
model_inputs = tokenizer(prompt, return_tensors='pt')
model_inputs = {key: value.to(model_device) for key, value in model_inputs.items()}
attention_mask = model_inputs.get('attention_mask', torch.ones_like(model_inputs['input_ids']))

with torch.no_grad():
    prefill_outputs = model(**model_inputs, use_cache=True)

cache = prefill_outputs.past_key_values
cache_summary = summarize_past_key_values(cache)
cache_summary_path = ARTIFACT_DIR / 'prefill_cache_summary.json'
cache_summary_path.write_text(json.dumps(cache_summary, indent=2), encoding='utf-8')

next_token = prefill_outputs.logits[:, -1:].argmax(dim=-1)
decode_attention_mask = torch.cat(
    [
        attention_mask,
        torch.ones(
            (attention_mask.shape[0], 1),
            dtype=attention_mask.dtype,
            device=attention_mask.device,
        ),
    ],
    dim=1,
)
decode_inputs = {
    'input_ids': next_token,
    'attention_mask': decode_attention_mask,
    'past_key_values': cache,
    'use_cache': True,
}

with torch.no_grad():
    decode_hook_calls = capture_forward_io(
        deltanet_layer,
        lambda: model(**decode_inputs),
    )

decode_hook_path = ARTIFACT_DIR / 'decode_hook_summary.json'
decode_hook_path.write_text(json.dumps(decode_hook_calls, indent=2), encoding='utf-8')

print('cache_type:', type(cache))
print('cache_summary_count:', len(cache_summary))
for row in cache_summary:
    print(f"  {row['name']}: {row['shape']} {row['dtype']}")
print()
print('decode_hook_calls:')
pprint(decode_hook_calls)
print()
print('saved_cache_summary:', cache_summary_path.resolve())
print('saved_decode_hook_summary:', decode_hook_path.resolve())


cache_type: <class 'transformers.cache_utils.DynamicCache'>
cache_summary_count: 64
  layer 0.conv_states: [1, 8192, 4] torch.float16
  layer 0.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 1.conv_states: [1, 8192, 4] torch.float16
  layer 1.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 2.conv_states: [1, 8192, 4] torch.float16
  layer 2.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 3.keys: [1, 4, 1, 256] torch.float16
  layer 3.values: [1, 4, 1, 256] torch.float16
  layer 4.conv_states: [1, 8192, 4] torch.float16
  layer 4.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 5.conv_states: [1, 8192, 4] torch.float16
  layer 5.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 6.conv_states: [1, 8192, 4] torch.float16
  layer 6.recurrent_states: [1, 32, 128, 128] torch.float16
  layer 7.keys: [1, 4, 1, 256] torch.float16
  layer 7.values: [1, 4, 1, 256] torch.float16
  layer 8.conv_states: [1, 8192, 4] torch.float16
  layer 8.recurrent_st

## 4. Baseline Before / After FLA

In [6]:
def run_fresh_python(script: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, '-c', script],
        cwd=str(ROOT_DIR),
        capture_output=True,
        text=True,
        check=False,
    )


def save_process_log(name: str, completed: subprocess.CompletedProcess, command_text: str) -> Path:
    log_path = ARTIFACT_DIR / f'{name}.log'
    payload = [
        f'command: {command_text}',
        f'returncode: {completed.returncode}',
        '',
        '=== STDOUT ===',
        completed.stdout,
        '',
        '=== STDERR ===',
        completed.stderr,
    ]
    log_path.write_text('\n'.join(payload), encoding='utf-8')
    return log_path


def run_baseline_once(tag: str) -> tuple[dict, Path]:
    result_path = ARTIFACT_DIR / f'baseline_{tag}.json'
    script = textwrap.dedent(
        f'''
        import json
        import platform
        from pathlib import Path

        import torch

        from phase1_utils import MODEL_DIR, build_baseline_rows, load_model_and_tokenizer

        result_path = Path({json.dumps(str(result_path))})
        model, tokenizer, config = load_model_and_tokenizer(
            MODEL_DIR,
            torch_dtype=torch.float16,
            device_map='auto',
        )
        rows = build_baseline_rows(model, tokenizer, gen_tokens=64)
        payload = {{
            'python': platform.python_version(),
            'python_executable': __import__('sys').executable,
            'torch': torch.__version__,
            'cuda_available': torch.cuda.is_available(),
            'rows': rows,
        }}
        result_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print(json.dumps(payload, indent=2))
        '''
    )
    completed = run_fresh_python(script)
    log_path = save_process_log(
        f'baseline_{tag}',
        completed,
        f"{sys.executable} -c <baseline_{tag}>",
    )
    if completed.returncode != 0:
        raise RuntimeError(f'Baseline {tag} failed. See {log_path}')
    return json.loads(result_path.read_text(encoding='utf-8')), log_path


def install_fla() -> dict:
    attempts = [
        ('flash-linear-attention', [sys.executable, '-m', 'pip', 'install', 'flash-linear-attention']),
        ('fla-core', [sys.executable, '-m', 'pip', 'install', 'fla-core']),
    ]
    records = []
    installed_package = None
    for package_name, command in attempts:
        completed = subprocess.run(
            command,
            cwd=str(ROOT_DIR),
            capture_output=True,
            text=True,
            check=False,
        )
        log_path = save_process_log(
            f'install_{package_name}',
            completed,
            ' '.join(command),
        )
        record = {
            'package': package_name,
            'returncode': completed.returncode,
            'log_path': str(log_path.resolve()),
        }
        records.append(record)
        if completed.returncode == 0:
            installed_package = package_name
            break
    return {
        'installed_package': installed_package,
        'records': records,
    }


def collect_fla_source() -> tuple[dict, Path]:
    result_path = ARTIFACT_DIR / 'fla_fused_recurrent_info.json'
    source_copy_path = ARTIFACT_DIR / 'fla_fused_recurrent_source.py'
    script = textwrap.dedent(
        f'''
        import inspect
        import json
        from pathlib import Path

        import fla
        from fla.ops.gated_delta_rule import fused_recurrent_gated_delta_rule

        result_path = Path({json.dumps(str(result_path))})
        source_copy_path = Path({json.dumps(str(source_copy_path))})

        module_file = Path(inspect.getfile(fused_recurrent_gated_delta_rule))
        source_text = module_file.read_text(encoding='utf-8')
        source_copy_path.write_text(source_text, encoding='utf-8')
        payload = {{
            'module_file': str(module_file),
            'saved_copy': str(source_copy_path),
            'line_count': len(source_text.splitlines()),
        }}
        result_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        print(json.dumps(payload, indent=2))
        '''
    )
    completed = run_fresh_python(script)
    log_path = save_process_log(
        'fla_fused_recurrent_source',
        completed,
        f"{sys.executable} -c <fla_fused_recurrent_source>",
    )
    if completed.returncode != 0:
        raise RuntimeError(f'Collecting fla source failed. See {log_path}')
    return json.loads(result_path.read_text(encoding='utf-8')), log_path


baseline_before_path = ARTIFACT_DIR / 'baseline_before_fla.json'
before_log_path = ARTIFACT_DIR / 'baseline_before_fla.log'
if baseline_before_path.exists() and before_log_path.exists():
    baseline_before = json.loads(baseline_before_path.read_text(encoding='utf-8'))
else:
    baseline_before, before_log_path = run_baseline_once('before_fla')
print('baseline_before:')
pprint(baseline_before['rows'])
print('before_log_path:', before_log_path.resolve())
print()

install_log_path = ARTIFACT_DIR / 'install_flash-linear-attention.log'
if install_log_path.exists() and importlib.util.find_spec('fla') is not None:
    install_info = {
        'installed_package': 'flash-linear-attention',
        'records': [
            {
                'package': 'flash-linear-attention',
                'returncode': 0,
                'log_path': str(install_log_path.resolve()),
            }
        ],
    }
else:
    install_info = install_fla()
print('install_info:')
pprint(install_info)
print()

baseline_after = None
after_log_path = ARTIFACT_DIR / 'baseline_after_fla.log'
baseline_after_path = ARTIFACT_DIR / 'baseline_after_fla.json'
fla_source_info = None
fla_source_log_path = ARTIFACT_DIR / 'fla_fused_recurrent_source.log'
fla_source_info_path = ARTIFACT_DIR / 'fla_fused_recurrent_info.json'
if baseline_after_path.exists() and after_log_path.exists():
    baseline_after = json.loads(baseline_after_path.read_text(encoding='utf-8'))
if fla_source_info_path.exists() and fla_source_log_path.exists():
    fla_source_info = json.loads(fla_source_info_path.read_text(encoding='utf-8'))

if baseline_after is None and (install_info['installed_package'] is not None or importlib.util.find_spec('fla') is not None):
    baseline_after, after_log_path = run_baseline_once('after_fla')
if fla_source_info is None and (install_info['installed_package'] is not None or importlib.util.find_spec('fla') is not None):
    fla_source_info, fla_source_log_path = collect_fla_source()

if baseline_after is not None:
    print('baseline_after:')
    pprint(baseline_after['rows'])
    print('after_log_path:', after_log_path.resolve())
    print()

if fla_source_info is not None:
    print('fla_source_info:')
    pprint(fla_source_info)
    print('fla_source_log_path:', fla_source_log_path.resolve())
elif install_info['installed_package'] is None and importlib.util.find_spec('fla') is None:
    print('Skipping post-install benchmark because no fla package could be installed.')

comparison_path = ARTIFACT_DIR / 'baseline_compare.json'
comparison_payload = {
    'before_log_path': str(before_log_path.resolve()),
    'before_fallback_warning': 'Falling back to torch implementation' in before_log_path.read_text(encoding='utf-8'),
}

if baseline_after is not None and after_log_path is not None:
    before_rows = {row['scenario']: row for row in baseline_before['rows']}
    compare_rows = []
    for row in baseline_after['rows']:
        before_row = before_rows[row['scenario']]
        speedup = before_row['decode_latency_s_per_token'] / row['decode_latency_s_per_token']
        compare_rows.append(
            {
                'scenario': row['scenario'],
                'before_decode_latency_s_per_token': before_row['decode_latency_s_per_token'],
                'after_decode_latency_s_per_token': row['decode_latency_s_per_token'],
                'decode_speedup': speedup,
                'before_prefill_latency_s': before_row['prefill_latency_s'],
                'after_prefill_latency_s': row['prefill_latency_s'],
            }
        )
    comparison_payload.update(
        {
            'after_log_path': str(after_log_path.resolve()),
            'after_fallback_warning': 'Falling back to torch implementation' in after_log_path.read_text(encoding='utf-8'),
            'compare_rows': compare_rows,
            'fla_source_info': fla_source_info,
            'fla_source_log_path': str(fla_source_log_path.resolve()) if fla_source_log_path is not None else None,
        }
    )
    print('baseline_compare:')
    pprint(compare_rows)

comparison_path.write_text(json.dumps(comparison_payload, indent=2), encoding='utf-8')
print('saved_compare_payload:', comparison_path.resolve())


baseline_before:
[{'decode_latency_s_per_token': 0.04317721108124957,
  'input_tokens': 133,
  'peak_vram_mib': 8739.12548828125,
  'prefill_latency_s': 0.2361678029000359,
  'scenario': 'short_128'},
 {'decode_latency_s_per_token': 0.044513491971875395,
  'input_tokens': 527,
  'peak_vram_mib': 8826.06982421875,
  'prefill_latency_s': 0.2863386720000108,
  'scenario': 'mid_512'},
 {'decode_latency_s_per_token': 0.04301990975625074,
  'input_tokens': 2060,
  'peak_vram_mib': 9182.87646484375,
  'prefill_latency_s': 0.6157379737999917,
  'scenario': 'long_2048'}]
before_log_path: /home/haozhong/ECE9483/artifacts/deltanet_diagnostics/baseline_before_fla.log

install_info:
{'installed_package': 'flash-linear-attention',
 'records': [{'log_path': '/home/haozhong/ECE9483/artifacts/deltanet_diagnostics/install_flash-linear-attention.log',
              'package': 'flash-linear-attention',
              'returncode': 0}]}

baseline_after:
[{'decode_latency_s_per_token': 0.03481940757187445,
 

## 5. Saved Artifacts

In [7]:
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path.name)


baseline_after_fla.json
baseline_after_fla.log
baseline_before_fla.json
baseline_before_fla.log
baseline_compare.json
decode_hook_summary.json
deltanet_layer_structure.txt
deltanet_layer_tensors.json
fla_fused_recurrent_info.json
fla_fused_recurrent_source.log
fla_fused_recurrent_source.py
install_flash-linear-attention.log
prefill_cache_summary.json
transformers_qwen3_5_modeling_qwen3_5.py
